In [0]:
%sql
use catalog workspace;
use schema customer360;

select current_database(), current_catalog(), current_schema();


In [0]:
%python
raw_custData = [
    (1001, "  john smith  ", "214-555-1001", "JOHN@EMAIL.COM", "dallas", "TX", "Wireless", 89.99),
    (1002, "MARY JONES", "(469) 555-1002", "mary@email.com ", "PLANO", "tx", "Fiber", 129.99),
    (1003, " Steve Brown ", "972.555.1003", None, "frisco", "Tx", "Wireless", 75.50),
    (1004, "sarah wilson", "214 555 1004", "SARAH@EMAIL.COM", "Irving", "TX", "Fiber", None),
    (1005, "DAVID LEE", None, " david@email.com", "mckinney", "tx", "Wireless", 95.00)
]

raw_columns = [
    "custId",
    "custName",
    "phone",
    "email",
    "city",
    "state",
    "product",
    "monthlyCharge"
    ]

dfRawCustomers = spark.createDataFrame(raw_custData, raw_columns)
dfRawCustomers.write.mode("overwrite").saveAsTable("customers_raw")

display(dfRawCustomers)

### Clean the customerName
- **trim()** --> remove whiteSpaces
- **initcap()** --> do Camel case
- **col()**--> identifies on which column the transformation needs to be applied. 

- **mergeschema()**--> updates the variable with the changes as dataframes are immutable, _withColumn() returns a new DataFrame; it doesn't mutate the original DataFrame in place._

In [0]:
%python

from pyspark.sql.functions import *

# Step 1: Clean customer name
dfCleanCustName = dfRawCustomers.withColumn("custName", trim(initcap(col("custName"))))

display(dfCleanCustName)
                                                            

In [0]:
%sql
select * from customer360.customers_raw;

### Standardize Email
Issues:
- Whitespaces in email value
- Multi case ine  email

In [0]:
from pyspark.sql.functions import *

# Step 2: Clean email (chain from previous step)
dfCleanEmail = dfCleanCustName.withColumn("email", trim(lower(col("email"))))

display(dfCleanEmail)

### Standardize City and State
- **City**  → proper case --> first letter caps
- **State** → uppercase

In [0]:
%python
from pyspark.sql.functions import *

# Step 3: Clean city (chain from previous step)
dfCleanCity = dfCleanEmail.withColumn("city", trim(initcap(col("city"))))

# Step 4: Clean state (chain from city cleaning)
dfCleanState = dfCleanCity.withColumn("state", trim(upper(col("state"))))

# Now save the fully cleaned data
dfCleanState.write.mode("overwrite").option("mergeSchema","true").saveAsTable("customers_raw")

display(dfCleanState)


### Phone-Number cleaning
- use **_regexp_replace()_** --. to clean /substitute data

In [0]:
%python
dfCleanPh = dfCleanState.withColumn("phone", regexp_replace(col("phone") ,"[^0-9]",""))
                         
dfCleanPh.write.mode("overwrite").option("mergeSchema","true").saveAsTable("customers_raw")


### Handling NULL values
- fillna() --> subistute null with a value
- coalesce() --> 


In [0]:
%python
from pyspark.sql.functions import *

dfCleanHandleNull = dfCleanPh.withColumn("email", coalesce(col("email"), lit("unknown"))) \
                             .withColumn("monthlyCharge", coalesce(col("monthlyCharge"), lit(0.0)))

dfCleanHandleNull.write.mode("overwrite").option("mergeSchema","true").saveAsTable("customers_raw")

display(dfCleanHandleNull)


### 

### Add derived Columns

In [0]:
%python
from pyspark.sql.functions import *

dfAddDericedCol = dfCleanHandleNull.withColumn("AnnualCharge", round(col("monthlyCharge") * 12, 2))

dfAddDericedCol.write.mode("overwrite").option("mergeSchema","true").saveAsTable("customers_raw")

In [0]:
%sql
select * from customer360.customers_raw;

### Create Silver table 

In [0]:
%python
from pyspark.sql.functions import *

dfCustomerSilver = dfAddDericedCol.write.mode("overwrite").option("mergeSchema","true").saveAsTable("customers_clean")

display(dfCustomerSilver)




In [0]:
%sql
select * from customers_clean;